# Post-processing LSTM results

After running notebook 11 (full LSTM grid) and notebook 12 (MIL LSTM), this
notebook recomputes the downstream analyses that depend on the LSTM
predictions:

1. **Cloud-cover stratification** — Sentinel-1's marginal value across
   per-year tercile and median-cut strata of Sentinel-2 observation rate
2. **Per-year LOYO stability** — macro MAE per stream per held-out year
3. **Supplementary Table S6** — per-year cloud-conditional Δ
4. **Figure regeneration** — F1 through F6, F8, F9 using the new LSTM CSV

The cloud-cover stratification reuses the existing per-field
`per_field_s2_strata.csv` from `phase5_outputs/`; only the underlying
predictions change.


## 1. Setup + load the new LSTM results

In [ ]:
from pathlib import Path
import sys, os
PROJECT_ROOT = Path('/fs1/home/atikul/Phenology')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

LSTM_DIR = PROJECT_ROOT / 'phase3_outputs' / 'lstm_v2'
CLOUD_DIR = PROJECT_ROOT / 'phase5_outputs'
OUT_DIR = PROJECT_ROOT / 'phase5_outputs_lstm'
OUT_DIR.mkdir(parents=True, exist_ok=True)

results = pd.read_csv(LSTM_DIR / 'lstm_v2_results.csv')
print(f'Loaded {len(results)} rows from lstm_v2_results.csv')
print(f'Streams: {sorted(results["stream"].unique())}')
print(f'Policies: {sorted(results["policy"].unique())}')
print(f'Years: {sorted(results["test_year"].unique())}')
print(f'Phases: {sorted(results["phase"].unique())}')


## 2. Cloud-cover stratification — per-year terciles

In [ ]:
PHASES = [10, 12, 15, 18, 19, 21, 24]

# Per-field S2 observation rate strata (from earlier project work)
pf = pd.read_csv(CLOUD_DIR / 'per_field_s2_strata.csv')
pf = pf.drop_duplicates(subset=['xid', 'year'])

def py_strat(group):
    lo = group.quantile(1/3); hi = group.quantile(2/3)
    return pd.cut(group, [-np.inf, lo, hi, np.inf], labels=['low', 'med', 'high'])

pf['stratum_pyt'] = (pf.groupby('year', observed=True)['s2_obs_rate']
                        .transform(py_strat))
print(f'Per-field strata: {pf["stratum_pyt"].value_counts().to_dict()}')

# Load per-record predictions from the LSTM npz
z = np.load(LSTM_DIR / 'lstm_v2_predictions.npz', allow_pickle=False)
print(f'\nKeys in predictions npz: {len(z.files)} entries')
print(f'  (one per (stream, policy, year) combination, with subkeys yt/yp/xid/etc.)')


In [ ]:
# Build per-record error frame for S2_only and S1+S2 streams
rows = []
for stream in ['S2_only', 'S1+S2']:
    for y in [2017, 2018, 2019, 2020, 2021]:
        pref = f'{stream}__all10__{y}'
        try:
            yt = z[f'{pref}__yt']; yp = z[f'{pref}__yp']; xid = z[f'{pref}__xid']
        except KeyError:
            print(f'WARN: missing {pref}')
            continue
        for j, phase in enumerate(PHASES):
            valid = (yt[:, j] >= 0) & (yp[:, j] >= 0)
            for i in np.where(valid)[0]:
                rows.append({'stream': stream, 'phase': int(phase),
                              'year': int(y), 'xid': int(xid[i]),
                              'ae': float(abs(yt[i, j] - yp[i, j]))})
errs = pd.DataFrame(rows)
print(f'Per-record error frame: {len(errs)} rows')

errs = errs.merge(pf[['xid', 'year', 'stratum_pyt']], on=['xid', 'year'], how='left')
errs = errs.dropna(subset=['stratum_pyt'])
print(f'After merge with strata: {len(errs)} rows')


In [ ]:
# Aggregate MAE by (phase, stream, stratum)
agg = (errs.groupby(['phase', 'stream', 'stratum_pyt'], observed=True)['ae']
            .mean().unstack('stream'))

# S1's marginal value = MAE(S2_only) - MAE(S1+S2)
s1_marg = (agg['S2_only'] - agg['S1+S2']).unstack('stratum_pyt')
s1_marg = s1_marg[['low', 'med', 'high']]
s1_marg = s1_marg.reindex(PHASES)
s1_marg.columns = ['most_cloudy', 'medium', 'least_cloudy']
s1_marg['Δ_cloudy_minus_clear'] = s1_marg['most_cloudy'] - s1_marg['least_cloudy']

print('=== Sentinel-1 marginal value (days), per-year terciles, LSTM ===')
print('Positive means adding S1 reduces MAE.')
print(s1_marg.round(2).to_string())

s1_marg.to_csv(OUT_DIR / 'cloud_strata_pyt_lstm.csv')


## 3. Per-year cloud-conditional Δ (for Supplementary Table S6)

In [ ]:
# Per-year, cloudy-minus-clear difference of S1 marginal value
agg_year = (errs.groupby(['phase', 'stream', 'stratum_pyt', 'year'], observed=True)['ae']
                .mean().unstack('stream'))
s1_marg_year = (agg_year['S2_only'] - agg_year['S1+S2']).unstack('stratum_pyt')
delta_year = (s1_marg_year['low'] - s1_marg_year['high']).unstack('year')
delta_year = delta_year.reindex(PHASES)
delta_year['mean'] = delta_year.mean(axis=1)
delta_year['years_positive'] = (delta_year[[2017,2018,2019,2020,2021]] > 0).sum(axis=1)

print('=== Cloudy minus clear Δ per (phase, year), LSTM ===')
print(delta_year.round(2).to_string())

delta_year.to_csv(OUT_DIR / 'cloud_strata_per_year_lstm.csv')


## 4. Per-year LOYO stability (Table 3 numbers for v5)

In [ ]:
sub = results[results['policy'] == 'all10']
per_year = (sub.groupby(['test_year', 'stream'])['mae'].mean()
                .unstack('stream')
                .reindex(['S1_only','S2_only','Met_only','S1+S2','All'], axis=1))
per_year.loc['mean']  = per_year.mean(axis=0)
per_year['spread']    = per_year[['S1_only','S2_only','Met_only','S1+S2','All']].max(axis=1) - \
                          per_year[['S1_only','S2_only','Met_only','S1+S2','All']].min(axis=1)
print('=== Macro MAE per test year × stream (all10, LSTM) ===')
print(per_year.round(2).to_string())
per_year.to_csv(OUT_DIR / 'per_year_loyo_lstm.csv')


## 5. Supervision policy comparison (Table 1 and Figure 4)

In [ ]:
# Per (stream, policy) macro MAE
pol = (results.groupby(['stream','policy'])['mae'].mean()
                .unstack('policy'))
pol['Δ_closest_minus_shared'] = pol['rank1'] - pol['all10']
print('=== Policy comparison: macro MAE, LSTM ===')
print(pol.round(3).to_string())
pol.to_csv(OUT_DIR / 'policy_compare_lstm.csv')

# Win rate per phase (Figure 4 left panel)
folds = (results.groupby(['phase', 'stream', 'policy', 'test_year'])['mae']
                  .first().reset_index())
pivot = folds.pivot_table(index=['phase', 'stream', 'test_year'],
                            columns='policy', values='mae')
pivot['winner'] = np.where(pivot['all10'] < pivot['rank1'], 'all10', 'rank1')
win_rate = (pivot.reset_index().groupby('phase')['winner']
                  .apply(lambda s: (s == 'all10').mean()))
print('\n=== Shared-label win rate per phase, LSTM ===')
for phase, wr in win_rate.items():
    print(f'  phase {int(phase)}: {wr*100:.0f}%')
win_rate.to_csv(OUT_DIR / 'policy_winrate_lstm.csv')


## 6. Regenerate all figures

In [ ]:
# The figures module looks for predictions at:
#   phase3_outputs/lstm/lstm_results.csv      (we now have lstm_v2_results.csv)
#   phase3_outputs/lstm/lstm_predictions.npz  (we now have lstm_v2_predictions.npz)
#   phase3_outputs/lstm/lstm_logits.npz       (we saved this in notebook 11)
#
# We can either point the figures module at the new files, or symlink.
# Simplest: temporarily replace the old files with the new ones for figure generation.

import shutil

OLD_LSTM_DIR = PROJECT_ROOT / 'phase3_outputs' / 'lstm'
# Back up old GRU files (rename so they're preserved)
for fn in ['lstm_results.csv', 'lstm_predictions.npz']:
    src = OLD_LSTM_DIR / fn
    if src.exists() and not (OLD_LSTM_DIR / f'{fn}.gru_backup').exists():
        shutil.copy(src, OLD_LSTM_DIR / f'{fn}.gru_backup')
        print(f'Backed up {fn} -> {fn}.gru_backup')

# Copy LSTM v2 files into the location the figures module expects
shutil.copy(LSTM_DIR / 'lstm_v2_results.csv', OLD_LSTM_DIR / 'lstm_results.csv')
shutil.copy(LSTM_DIR / 'lstm_v2_predictions.npz', OLD_LSTM_DIR / 'lstm_predictions.npz')
if (LSTM_DIR / 'lstm_logits.npz').exists():
    shutil.copy(LSTM_DIR / 'lstm_logits.npz', OLD_LSTM_DIR / 'lstm_logits.npz')
print('Promoted LSTM v2 files to canonical location for figure generation.')

# Also promote the MIL LSTM file if available
MIL_LSTM_NPZ = PROJECT_ROOT / 'phase4_outputs_lstm' / 'mil_partA_predictions_lstm.npz'
MIL_DEST     = PROJECT_ROOT / 'phase4_outputs' / 'mil_partA_predictions.npz'
if MIL_LSTM_NPZ.exists():
    if MIL_DEST.exists() and not (MIL_DEST.parent / 'mil_partA_predictions.npz.gru_backup').exists():
        shutil.copy(MIL_DEST, MIL_DEST.parent / 'mil_partA_predictions.npz.gru_backup')
    shutil.copy(MIL_LSTM_NPZ, MIL_DEST)
    print('Promoted MIL LSTM npz to canonical location.')


In [ ]:
# Now run the figures pipeline
import importlib
import figures, figures.figures, figures.style
importlib.reload(figures.style)
importlib.reload(figures.figures)
importlib.reload(figures)

import matplotlib
matplotlib.use('Agg')

from figures import make_all

results = make_all(
    data_dir=PROJECT_ROOT,
    output_dir=PROJECT_ROOT / 'phase6_figures',
)
print()
print(results.to_string(index=False))


## 7. Preview the regenerated figures

In [ ]:
from IPython.display import Image, display
fig_dir = PROJECT_ROOT / 'phase6_figures'
for png in sorted(fig_dir.glob('*.png')):
    print(f'\n--- {png.name} ---')
    display(Image(filename=str(png), width=700))


---

## Done

Paste back to me:

1. The **cloud-conditional Δ table** (Section 2 last cell)
2. The **per-year cloud Δ table** (Section 3)
3. The **per-year LOYO stability table** (Section 4)
4. The **policy comparison** + **win rate** (Section 5)

I'll use those to fill in tables 1, 2, 3, S6 in manuscript v5.

If any figure looks off in Section 7, tell me which and what changed (e.g. axis
ranges look weird, a bar is now hidden, etc.) and I'll patch the figures module.
